# prepare PATH

In [2]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
# step up the dir tree from this notebook to find the root dir of this project.
while not (repo_root / "src" / "__init__.py").is_file():
    parent = repo_root.parent
    if parent == repo_root:
        raise RuntimeError(
            "Could not find the repository root from the notebook's working directory."
        )
    repo_root = parent

sys.path.insert(0, str(repo_root))

In [3]:
# import system libraries
import torch
from copy import deepcopy

# import user-defined libraries
from src.prepared_data import load_entity

# import data processing libraries
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# simple implementation flow
- [ ] load data into train, validation and test splits.
- [ ] windowize into train, validation and test windows. write a custom pytorch dataset and dataloader.
- [ ] write the training loop and tracking the best checkpoint using a metric.
- [ ] write the model.
- [ ] evaluate on test set, which means calculate `VUS-PR`, `VUS-ROC`, `Affiliation F1`, etc.

## load data

In [ ]:
# data = load_entity(
#     Path(
#         "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/"
#     ),
#     "swat"
# )
# train = data["splits"]["train"]
# validation = data["splits"]["validation"]
# test = data["splits"]["validation"]
# labels = train["labels"]

In [ ]:
# print(len(train['x']))
# print(len(validation['x']))
# print(len(test['x']))

396000
99000
99000


In [ ]:
# # inspect keys of each split object
# data['splits']['train'].keys()

dict_keys(['x', 'labels', 'source_file', 'source_row', 'segments', 'identity'])

## dataset

In [7]:
train_table = pq.read_table(
    "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/swat/train.parquet"
)
validation_table = pq.read_table(
    "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/swat/validation.parquet"
)
test_table = pq.read_table(
    "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/swat/test.parquet"
)

In [ ]:
# type(train_table.slice(offset=0, length=100))

In [ ]:
# arrays_dict = {
#     name: col.to_numpy() for name, col in zip(train_table.column_names,
#                                               train_table.columns)
# }
# arrays_dict

In [12]:
arr_2d = np.column_stack([col.to_numpy() for col in train_table.columns])
arr_2d.shape

(396000, 48)

In [ ]:
class SWaTDataset(torch.utils.data.Dataset):
    def __init__(self, split, stride, win_size, parquet_file, root_dir, transform=None):
        """
        Arguments:
            parquet_file (string): Path to the parquet file with labels.
            root_dir (string): Directory with all signals.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.split = split
        self.strid = stride
        self.win_size = win_size
        self.parquet_file = (
            parquet_file  # this file can be train, validation or test file.
        )
        self.root_dir = root_dir
        self.transform = transform

        # load series of signals
        self.table: pa.Table = pq.read_table(parquet_file)
        self.X = self.table.select(self.table.column_names[:-4])

        # if this is test split, then load series of labels
        if split == "test":
            self.Y = self.table.select(self.table.column_names[-2])

    def __len__(self):
        return len(self.table) - self.win_size

    def __getitem__(self, idx):
        """
        Get the window starting at global index `idx` in `self.table`.
        """
        if idx > len(self.table) - self.win_size - 1:
            raise ValueError(
                "`idx` must not be after the starting point of the last window."
            )

        sample = None
        if self.split == "train":
            pa_slice = self.X.slice(offset=idx, length=self.win_size)
            np2darray = np.column_stack([col for col in pa_slice.columns])
            sample = {"win_values": np2darray}  # this is a window, which can be
            # multivariate or univariate.
        return sample

## coding GRN-v3

In [ ]:
# # input feature vectors are column vectors by convention.
# x = torch.Tensor(np.random.rand(3, 100, 48))
# d_model = x.shape[2]
# g_0 = torch.zeros((3, 100, 48))
# f_0 = x
# g_1 = f_0 + g_0
# g_1.shape

torch.Size([3, 100, 48])

In [ ]:
# # residual function 1
# f_1 = torch.nn.Conv1d(in_channels=100,
#                       out_channels=100,
#                       kernel_size=5,
#                       padding='same')
# # residual block t=1 output
# g_2 = f_1(g_1) + g_1
# g_2.shape

torch.Size([3, 100, 48])

In [ ]:
# # residual function 2
# f_2 = torch.nn.MultiheadAttention(embed_dim=48, num_heads=6)
# # residual block t=2 output
# # https://docs.pytorch.org/docs/2.15/generated/torch.nn.MultiheadAttention.html
# attn_output, attn_output_weights = f_2(g_2, g_2, g_2)
# # output of residual block t=2
# g_3 = attn_output + g_2
# g_3.shape

torch.Size([3, 100, 48])

In [ ]:
# b_4 = torch.ones((1, 48, 4))
# # batch first, then length of window, then number of dimensions, then number of layers.
# G_4 = torch.stack([g_0, g_1, g_2, g_3]).permute(1,2,3,0)
# # ...

torch.Size([3, 100, 48, 4])

In [ ]:
# class SingletonMeta(type):
#     """
#     https://refactoring.guru/design-patterns/singleton/python/example
#     The Singleton class can be implemented in different ways in Python. Some
#     possible methods include: base class, decorator, metaclass. We will use the
#     metaclass because it is best suited for this purpose.
#     """

#     _instances = {}

#     def __call__(cls, *args, **kwargs):
#         """
#         Possible changes to the value of the `__init__` argument do not affect
#         the returned instance.
#         """
#         if cls not in cls._instances:
#             instance = super().__call__(*args, **kwargs)
#             cls._instances[cls] = instance
#         return cls._instances[cls]


# class GRNv3(SingletonMeta, torch.nn.Module):
#     """
#     This class implements the Generalized Residual Network version 3
#     introduced within this paper https://arxiv.org/abs/2502.06785v1.

#     Target: Centralize all memory and computation overhead of GRN into this class.
#     This helps us profile and analyze more easily later.

#     Target: I want this class to be Singleton.
#     """

#     def __init__(self, hidden_dim):
#         """
#         This __init__method should be called once and only once in
#         whole DeepCrossAttention model.
#         """
#         pass

#     def forward(self, x):
#         pass

In [ ]:
# class DeepCrossAttention(torch.nn.Module):
#     def __init__(self, hidden_dim):
#         self.d_model = hidden_dim

#         # self.g_0 = torch.zeros((hidden_dim, 1))
#         self.G_t = []
#         self.b_t = torch.ones()
#         self.w_t = torch.zeros()
#         self.grn_activation = torch.nn.ReLU()

#     def forward(self, x):
#         """
#         Arguments:
#             x: [B, T, C] (torch.Tensor):    input batch of time-series window(s).
#                                             or time-series feature matrice(s).
#         """
#         if len(self.G_t) == 0:
#             self.G_t.append(x) # f_0(g_0) = x

#         layers_outputs = torch.stack(self.G_t, dim=0)   # [B, d_model, l]
#         w_bar = torch.ones((self.d_model, self.d_model)) \
#                 @ self.grn_activation(self.w_t.T @ layers_outputs)
#         g_t = (layers_outputs * (self.b_t + w_bar)) @ torch.ones((len(self.G_t), 1))

In [ ]:
# class MustardNetwork(torch.nn.Module):
#     def __init__(self):
#         pass

#     def forward(self, x):
#         """
#         x: [B,T,C]
#         """
#         pass

## training and validation

## evaluation